# Day 077 — Exercise 5: LiveVisionAgent

**What you'll build:** `LiveVisionAgent` — a context-manager camera agent that opens a camera, reads frames, and analyzes them with a vision LLM.

**Why it matters:** The context manager (`with` statement) guarantees the camera is always released — even if an exception occurs inside the `with` block. This is the standard Python pattern for any resource that must be explicitly closed.

In [ ]:
import numpy as np
from PIL import Image as _PILImage

def _make_mock_frame(h=100, w=100, val=50):
    return np.full((h, w, 3), val, dtype=np.uint8)

class _MockCap:
    def __init__(self, n=5, h=100, w=100):
        self._frames = [_make_mock_frame(h, w) for _ in range(n)]
        self._idx = 0
    def isOpened(self):
        return True
    def read(self):
        if self._idx >= len(self._frames):
            return False, None
        f = self._frames[self._idx]; self._idx += 1
        return True, f
    def release(self):
        pass
    def get(self, prop):
        return 0.0

_mock_camera_fn = lambda device: _MockCap(n=5)
_mock_analyze_fn = lambda img, q: 'FRAME:' + q[:12]
def open_camera(device=0, camera_fn=None):
    if camera_fn is not None:
        return camera_fn(device)
    import cv2
    cap = cv2.VideoCapture(device)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera device {device}')
    return cap

def read_frame(cap):
    return cap.read()
import io, base64

def frame_to_image(frame):
    from PIL import Image
    rgb = frame[:, :, ::-1]
    return Image.fromarray(rgb)

def analyze_frame(frame, question, analyze_fn=None):
    image = frame_to_image(frame)
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']
from pathlib import Path

def should_analyze(frame_count, every_n):
    return frame_count % every_n == 0

def save_frame(frame, path):
    out = Path(path)
    frame_to_image(frame).save(out, format='PNG')
    return out
def capture_frames(device=0, n_frames=5, every_n=1, camera_fn=None):
    cap = open_camera(device=device, camera_fn=camera_fn)
    frames = []
    frame_count = 0
    try:
        while len(frames) < n_frames:
            ret, frame = read_frame(cap)
            if not ret:
                break
            if should_analyze(frame_count, every_n):
                frames.append(frame)
            frame_count += 1
    finally:
        cap.release()
    return frames

def analyze_stream(device=0, task='Describe what you see.', n_frames=5,
                   every_n=1, camera_fn=None, analyze_fn=None):
    cap = open_camera(device=device, camera_fn=camera_fn)
    results = []
    frame_count = 0
    analyzed = 0
    try:
        while analyzed < n_frames:
            ret, frame = read_frame(cap)
            if not ret:
                break
            if should_analyze(frame_count, every_n):
                description = analyze_frame(frame, task, analyze_fn=analyze_fn)
                results.append({'frame_idx': frame_count, 'description': description})
                analyzed += 1
            frame_count += 1
    finally:
        cap.release()
    return results


## Task

Implement `LiveVisionAgent(device=0, camera_fn=None, analyze_fn=None)`:

- `__init__`: store device/camera_fn/analyze_fn; `_cap=None`, `_last_frame=None`
- `open(device=None)`: `d = device if device is not None else self._device`; `self._cap = open_camera(d, camera_fn=self._camera_fn)`; `return self`
- `read()`: RuntimeError if `_cap` is None; `ret, frame = read_frame(self._cap)`; store if ret; return frame or None
- `analyze(question, frame=None)`: frame-or-last guard (ValueError); `analyze_frame(f, question, analyze_fn=self._analyze_fn)`
- `describe(frame=None)`: `self.analyze('Describe what you see in detail.', frame=frame)`
- `save(path, frame=None)`: frame-or-last guard; `save_frame(f, path)`
- `close()`: `if self._cap: self._cap.release(); self._cap = None`
- `__enter__`: `self.open(); return self`
- `__exit__(*args)`: `self.close()`

## Your Implementation

In [ ]:
class LiveVisionAgent:
    """Context-manager camera agent with vision analysis."""

    def __init__(self, device=0, camera_fn=None, analyze_fn=None):
        raise NotImplementedError

    def open(self, device=None):
        raise NotImplementedError

    def read(self):
        raise NotImplementedError

    def analyze(self, question, frame=None):
        raise NotImplementedError

    def describe(self, frame=None):
        raise NotImplementedError

    def save(self, path, frame=None):
        raise NotImplementedError

    def close(self):
        raise NotImplementedError

    def __enter__(self):
        raise NotImplementedError

    def __exit__(self, *args):
        raise NotImplementedError


In [ ]:
class LiveVisionAgent:
    def __init__(self, device=0, camera_fn=None, analyze_fn=None):
        self._device = device
        self._camera_fn = camera_fn
        self._analyze_fn = analyze_fn
        self._cap = None
        self._last_frame = None

    def open(self, device=None):
        d = device if device is not None else self._device
        self._cap = open_camera(device=d, camera_fn=self._camera_fn)
        return self

    def read(self):
        if self._cap is None:
            raise RuntimeError('Camera not open: call open() first or use as context manager.')
        ret, frame = read_frame(self._cap)
        if ret:
            self._last_frame = frame
        return frame if ret else None

    def analyze(self, question, frame=None):
        f = frame if frame is not None else self._last_frame
        if f is None:
            raise ValueError('No frame: call read() first or pass frame.')
        return analyze_frame(f, question, analyze_fn=self._analyze_fn)

    def describe(self, frame=None):
        return self.analyze('Describe what you see in detail.', frame=frame)

    def save(self, path, frame=None):
        f = frame if frame is not None else self._last_frame
        if f is None:
            raise ValueError('No frame: call read() first or pass frame.')
        return save_frame(f, path)

    def close(self):
        if self._cap is not None:
            self._cap.release()
            self._cap = None

    def __enter__(self):
        self.open()
        return self

    def __exit__(self, *args):
        self.close()


## Automated checks

In [ ]:

score, total = 0, 6
try:
    import numpy as np, tempfile, os

    agent = LiveVisionAgent(camera_fn=_mock_camera_fn, analyze_fn=_mock_analyze_fn)
    agent.open()
    assert agent._cap is not None
    score += 1; print("✅ open() sets _cap")

    frame = agent.read()
    assert isinstance(frame, np.ndarray)
    assert agent._last_frame is not None
    score += 1; print("✅ read() returns ndarray and stores _last_frame")

    desc = agent.describe()
    assert isinstance(desc, str)
    score += 1; print("✅ describe() returns str using stored frame")

    a = agent.analyze('What color?')
    assert isinstance(a, str)
    score += 1; print("✅ analyze() returns str")

    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        tmp = f.name
    try:
        p = agent.save(tmp)
        assert os.path.getsize(tmp) > 0
        score += 1; print("✅ save() writes non-empty PNG")
    finally:
        os.unlink(tmp)

    agent.close()
    assert agent._cap is None
    score += 1; print("✅ close() releases cap and sets None")

    # Context manager
    with LiveVisionAgent(camera_fn=_mock_camera_fn, analyze_fn=_mock_analyze_fn) as a2:
        f2 = a2.read()
        assert isinstance(f2, np.ndarray)
    assert a2._cap is None, "cap should be released after with block"
    print("✅ context manager: cap released on __exit__")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class LiveVisionAgent:
    def __init__(self, device=0, camera_fn=None, analyze_fn=None):
        self._device = device
        self._camera_fn = camera_fn
        self._analyze_fn = analyze_fn
        self._cap = None
        self._last_frame = None

    def open(self, device=None):
        d = device if device is not None else self._device
        self._cap = open_camera(device=d, camera_fn=self._camera_fn)
        return self

    def read(self):
        if self._cap is None:
            raise RuntimeError('Camera not open: call open() first or use as context manager.')
        ret, frame = read_frame(self._cap)
        if ret:
            self._last_frame = frame
        return frame if ret else None

    def analyze(self, question, frame=None):
        f = frame if frame is not None else self._last_frame
        if f is None:
            raise ValueError('No frame: call read() first or pass frame.')
        return analyze_frame(f, question, analyze_fn=self._analyze_fn)

    def describe(self, frame=None):
        return self.analyze('Describe what you see in detail.', frame=frame)

    def save(self, path, frame=None):
        f = frame if frame is not None else self._last_frame
        if f is None:
            raise ValueError('No frame: call read() first or pass frame.')
        return save_frame(f, path)

    def close(self):
        if self._cap is not None:
            self._cap.release()
            self._cap = None

    def __enter__(self):
        self.open()
        return self

    def __exit__(self, *args):
        self.close()
```

**Why `f = frame if frame is not None else self._last_frame`** instead of `frame or self._last_frame`? A valid frame ndarray is truthy, but an explicit `is not None` check is clearer about intent and handles edge cases like a zero-value frame array.

</details>